# Análisis Discriminante Cuadrático (QDA) — Clasificador

**Repositorio:** `autodidact/classification/qda/`

---

## 1. Idea General e Intuición

**QDA** (Quadratic Discriminant Analysis) es la extensión natural de LDA que **relaja el supuesto de covarianza compartida**. Cada clase tiene su propia matriz de covarianza $\boldsymbol{\Sigma}_k$, lo que produce **fronteras de decisión cuadráticas** (elipses, hipérbolas, parábolas) en lugar de hiperplanos lineales.

**Analogía:** Si LDA dibuja una línea recta para separar clases, QDA dibuja curvas. Esto lo hace más flexible pero también requiere estimar más parámetros.

**Cuándo QDA supera a LDA:**
- Cuando las covarianzas de las clases son **significativamente distintas**.
- Cuando las clases tienen **formas elípticas diferentes** en el espacio de features.

## 2. Fundamento Matemático

### 2.1 Modelo Generativo

Cada clase $C_k$ tiene su **propia distribución gaussiana**:

$$p(\mathbf{x} \mid y=k) = \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

### 2.2 Función Discriminante Cuadrática

Aplicando el teorema de Bayes y tomando logaritmo:

$$\delta_k(\mathbf{x}) = -\frac{1}{2}\log|\boldsymbol{\Sigma}_k| - \frac{1}{2}(\mathbf{x}-\boldsymbol{\mu}_k)^\top\boldsymbol{\Sigma}_k^{-1}(\mathbf{x}-\boldsymbol{\mu}_k) + \log\pi_k$$

La clase predicha:
$$\hat{y} = \underset{k}{\text{argmax}} \; \delta_k(\mathbf{x})$$

El término $(\mathbf{x}-\boldsymbol{\mu}_k)^\top\boldsymbol{\Sigma}_k^{-1}(\mathbf{x}-\boldsymbol{\mu}_k)$ es la **distancia de Mahalanobis** al cuadrado desde $\mathbf{x}$ hasta la clase $k$.

La función es **cuadrática** en $\mathbf{x}$ porque $\boldsymbol{\Sigma}_k$ varía por clase (no cancela el término cuadrático como en LDA).

### 2.3 Parámetros Estimados

$$\hat{\boldsymbol{\mu}}_k = \frac{1}{n_k}\sum_{i: y_i=k}\mathbf{x}_i$$

$$\hat{\boldsymbol{\Sigma}}_k = \frac{1}{n_k - 1}\sum_{i: y_i=k}(\mathbf{x}_i - \hat{\boldsymbol{\mu}}_k)(\mathbf{x}_i - \hat{\boldsymbol{\mu}}_k)^\top$$

### 2.4 Comparativa LDA vs QDA

| Aspecto | **LDA** | **QDA** |
|---|---|---|
| Covarianza | Compartida $\boldsymbol{\Sigma}$ | Por clase $\boldsymbol{\Sigma}_k$ |
| Frontera | **Lineal** (hiperplano) | **Cuadrática** (cónica) |
| Parámetros | $O(d^2 + Kd)$ | $O(Kd^2 + Kd)$ |
| Sesgo | Mayor | Menor |
| Varianza | Menor | Mayor |
| Muestras mínimas | $n > d$ | $n_k > d$ por clase |
| Regularización | `shrinkage` | `reg_param` |

### 2.5 Distancia de Mahalanobis

$$D_M(\mathbf{x}, \boldsymbol{\mu}_k) = \sqrt{(\mathbf{x}-\boldsymbol{\mu}_k)^\top\boldsymbol{\Sigma}_k^{-1}(\mathbf{x}-\boldsymbol{\mu}_k)}$$

Generaliza la distancia Euclidiana: tiene en cuenta la correlación y la escala de las features. Un punto se asigna a la clase con menor distancia de Mahalanobis (ajustada por el prior).

## 3. ¿Cuándo Usarlo y Cuándo NO?

### ✅ Úsalo cuando:
- Las **covarianzas de las clases son claramente distintas**.
- Tienes **suficientes muestras por clase** ($n_k > d$, idealmente $n_k \gg d$).
- Las distribuciones de clase son **aproximadamente gaussianas**.
- Necesitas un clasificador **rápido** con fronteras no lineales.
- Como **baseline probabilístico no lineal** antes de modelos más complejos.

### ❌ NO lo uses cuando:
- Pocas muestras por clase ($n_k \leq d$) — la covarianza por clase es singular.
- Las distribuciones son **muy no gaussianas** (multimodal, discreta).
- Alta dimensionalidad sin regularización — $K$ matrices de $d 	imes d$ son costosas.
- Las covarianzas son en realidad similares — LDA es suficiente y más estable.

## 4. Métricas de Evaluación

| Métrica | Notas para QDA |
|---|---|
| **Accuracy** | Clases balanceadas |
| **ROC-AUC** | QDA produce probabilidades bien calibradas si los supuestos se cumplen |
| **Log-loss** | Calibración de probabilidades gaussianas |
| **Brier Score** | Combina calibración y discriminación |

**Test de Box M** (diagnóstico de supuestos): contrasta $H_0: \boldsymbol{\Sigma}_1 = \cdots = \boldsymbol{\Sigma}_K$. Si se rechaza, QDA es preferible a LDA.

## 5. Problemas y Casos Raros

### 5.1 Singularidad de $\hat{\boldsymbol{\Sigma}}_k$
Si $n_k \leq d$, la covarianza de la clase $k$ es singular (no invertible).
**Solución:** `reg_param` en `QuadraticDiscriminantAnalysis` — añade $\rho\hat{\sigma}^2\mathbf{I}$ a cada covarianza:
$$\hat{\boldsymbol{\Sigma}}_k^{\text{reg}} = (1-\rho)\hat{\boldsymbol{\Sigma}}_k + \rho\hat{\sigma}^2\mathbf{I}$$

### 5.2 Alta Dimensionalidad
Con $d$ grande, estimar $K$ matrices de covarianza requiere muchas muestras.
**Regla práctica:** $n_k \geq 5d$ para estimaciones razonables.

### 5.3 Sobreajuste
QDA tiene más parámetros que LDA → más tendencia al sobreajuste.
**Solución:** `reg_param > 0` (interpolación entre QDA y LDA).

### 5.4 Outliers
La covarianza estimada por máxima verosimilitud es sensible a outliers extremos.
**Solución:** Estimadores robustos de covarianza (`MinCovDet` en sklearn).

## 6. Mejora de Resultados

| Parámetro | Efecto |
|---|---|
| `reg_param` | $\rho \in [0,1]$: 0=QDA puro, 1=LDA puro (interpolación) |
| `store_covariance` | Guardar $\hat{\Sigma}_k$ para inspección |
| `tol` | Tolerancia para detectar singularidades |

**`reg_param` como interpolación LDA↔QDA:**
- `reg_param=0.0`: QDA puro (cada clase con su covarianza).
- `reg_param=1.0`: equivalente a LDA (covarianza pooled).
- `reg_param=0.5`: intermedio, reduce varianza del estimador.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

from sklearn.datasets import load_iris, load_wine, load_breast_cancer, make_classification
from sklearn.discriminant_analysis import (
    QuadraticDiscriminantAnalysis, LinearDiscriminantAnalysis
)
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay, log_loss
)
from sklearn.covariance import EmpiricalCovariance

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
SEED = 42
rng  = np.random.RandomState(SEED)
print('✅ Librerías cargadas.')

---
## 7. Visualización: Frontera Lineal (LDA) vs Cuadrática (QDA)

In [ ]:
# =============================================================================
# DATOS SINTÉTICOS 2D: demostrar visualmente la diferencia en frontera de decisión
# Clase 0: covarianza circular | Clase 1: covarianza elíptica rotada
# =============================================================================
n = 200
# Clase 0: centrada en (-2, 0), covarianza isotrópica
cov0 = np.array([[1.0, 0.0], [0.0, 1.0]])
X0 = rng.multivariate_normal([-2, 0], cov0, n)

# Clase 1: centrada en (2, 0), covarianza elíptica rotada (diferente a clase 0)
cov1 = np.array([[2.5, 1.5], [1.5, 1.2]])
X1 = rng.multivariate_normal([2, 0], cov1, n)

X_syn = np.vstack([X0, X1])
y_syn = np.array([0]*n + [1]*n)

# Ajustar LDA y QDA
lda_syn = LinearDiscriminantAnalysis()
qda_syn = QuadraticDiscriminantAnalysis()
lda_syn.fit(X_syn, y_syn)
qda_syn.fit(X_syn, y_syn)

h = 0.05
x0r = [X_syn[:,0].min()-1, X_syn[:,0].max()+1]
x1r = [X_syn[:,1].min()-1, X_syn[:,1].max()+1]
xx, yy = np.meshgrid(np.arange(x0r[0], x0r[1], h),
                     np.arange(x1r[0], x1r[1], h))
grid = np.c_[xx.ravel(), yy.ravel()]

cmap_bg  = ListedColormap(['#FFAAAA', '#AAAAFF'])
cmap_pts = ListedColormap(['#CC0000', '#0000CC'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, model, title in zip(axes,
                              [lda_syn, qda_syn],
                              ['LDA — Frontera Lineal', 'QDA — Frontera Cuadrática']):
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_bg)
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    ax.scatter(X_syn[:,0], X_syn[:,1], c=y_syn, cmap=cmap_pts,
               edgecolors='k', s=20, alpha=0.6)
    acc = accuracy_score(y_syn, model.predict(X_syn))
    ax.set_title(f'{title}\nAccuracy={acc:.3f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')

plt.suptitle('LDA vs QDA: Covarianzas Distintas por Clase',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# VISUALIZACIÓN DE LAS ELIPSES DE COVARIANZA POR CLASE
# Las elipses muestran visualmente qué aprendió QDA sobre cada clase
# =============================================================================
from matplotlib.patches import Ellipse

def plot_cov_ellipse(cov, mean, ax, n_std=2, color='steelblue', alpha=0.3, label=''):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=theta,
                      alpha=alpha, color=color, label=label)
    ax.add_patch(ellipse)
    ax.scatter(*mean, color=color, s=100, marker='*', zorder=5, edgecolors='k')

fig, ax = plt.subplots(figsize=(8, 6))
colors_ell = ['crimson', 'steelblue']
labels_ell = ['Clase 0 (cov circular)', 'Clase 1 (cov elíptica)']
covs = [cov0, cov1]
means = [np.array([-2.0, 0.0]), np.array([2.0, 0.0])]

for cls in range(2):
    mask = y_syn == cls
    ax.scatter(X_syn[mask, 0], X_syn[mask, 1], color=colors_ell[cls],
               s=15, alpha=0.4, edgecolors='none')
    plot_cov_ellipse(qda_syn.covariance_[cls], qda_syn.means_[cls],
                     ax, n_std=2, color=colors_ell[cls], alpha=0.25,
                     label=f'Clase {cls} — ±2σ (QDA)')

ax.set_title('Elipses de Covarianza Aprendidas por QDA', fontsize=13, fontweight='bold')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 8. QDA en Datasets Reales

In [ ]:
# =============================================================================
# WINE DATASET — QDA con inspección de covarianzas por clase
# =============================================================================
wine = load_wine(as_frame=True)
X_w, y_w = wine.data.values, wine.target

X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(
    X_w, y_w, test_size=0.2, random_state=SEED, stratify=y_w
)
scaler_w = StandardScaler()
X_tr_sc  = scaler_w.fit_transform(X_tr_w)
X_te_sc  = scaler_w.transform(X_te_w)

qda_wine = QuadraticDiscriminantAnalysis(store_covariance=True)
qda_wine.fit(X_tr_sc, y_tr_w)

y_pred_w  = qda_wine.predict(X_te_sc)
y_proba_w = qda_wine.predict_proba(X_te_sc)

print(f'Accuracy : {accuracy_score(y_te_w, y_pred_w):.4f}')
print(f'Log-loss : {log_loss(y_te_w, y_proba_w):.4f}')
print()
print(classification_report(y_te_w, y_pred_w, target_names=wine.target_names))

In [ ]:
# =============================================================================
# DETERMINANTES DE LAS COVARIANZAS POR CLASE
# |Sigma_k| mide el "volumen" de la elipse de cada clase.
# Si son muy distintos → QDA está justificado sobre LDA.
# =============================================================================
print('Diagnóstico de covarianzas por clase:')
print(f'{"Clase":<12}  {"det(Sigma_k)":>14}  {"trace(Sigma_k)":>15}')
print('-' * 45)
for k, cls_name in enumerate(wine.target_names):
    cov_k = qda_wine.covariance_[k]
    det_k   = np.linalg.det(cov_k)
    trace_k = np.trace(cov_k)
    print(f'{cls_name:<12}  {det_k:>14.4f}  {trace_k:>15.4f}')

print('\nDiferencias en det(Sigma_k) → covarianzas distintas → QDA justificado.')

In [ ]:
# =============================================================================
# DISTANCIAS DE MAHALANOBIS POR CLASE
# Visualiza qué tan "lejos" está cada punto de test de cada clase
# =============================================================================
from scipy.spatial.distance import mahalanobis

mahal_dists = np.zeros((X_te_sc.shape[0], 3))
for k in range(3):
    cov_inv = np.linalg.pinv(qda_wine.covariance_[k])
    mu_k    = qda_wine.means_[k]
    for i, x in enumerate(X_te_sc):
        mahal_dists[i, k] = mahalanobis(x, mu_k, cov_inv)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors_w = ['#e74c3c', '#2ecc71', '#3498db']
for k, (ax, cls_name, color) in enumerate(zip(axes, wine.target_names, colors_w)):
    ax.hist(mahal_dists[y_te_w == k, k], bins=15, color=color,
            alpha=0.7, edgecolor='white', label=f'Clase correcta ({cls_name})')
    ax.hist(mahal_dists[y_te_w != k, k], bins=15, color='gray',
            alpha=0.5, edgecolor='white', label='Otras clases')
    ax.set_title(f'D. Mahalanobis a {cls_name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Distancia de Mahalanobis')
    ax.legend(fontsize=8)

plt.suptitle('Distancia de Mahalanobis por Clase — Wine Dataset',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Efecto de reg_param (Interpolación LDA ↔ QDA)

In [ ]:
# =============================================================================
# reg_param COMO DIAL ENTRE QDA (0) Y LDA (1)
# Útil cuando QDA sobreajusta por pocas muestras por clase
# =============================================================================
bc = load_breast_cancer(as_frame=True)
X_bc, y_bc = bc.data.values, bc.target

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

reg_values = np.linspace(0, 1, 21)
cv_roc, cv_acc = [], []

for rp in reg_values:
    pipe_rp = Pipeline([
        ('sc',  StandardScaler()),
        ('qda', QuadraticDiscriminantAnalysis(reg_param=rp))
    ])
    roc = cross_val_score(pipe_rp, X_bc, y_bc, cv=skf, scoring='roc_auc')
    acc = cross_val_score(pipe_rp, X_bc, y_bc, cv=skf, scoring='accuracy')
    cv_roc.append(roc.mean())
    cv_acc.append(acc.mean())

best_rp = reg_values[int(np.argmax(cv_roc))]
print(f'Mejor reg_param: {best_rp:.2f} | ROC-AUC: {max(cv_roc):.4f}')

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()
ax1.plot(reg_values, cv_roc, 'o-', color='steelblue', lw=2, ms=5, label='ROC-AUC')
ax2.plot(reg_values, cv_acc, 's--', color='seagreen', lw=1.5, ms=5, label='Accuracy')
ax1.axvline(best_rp, color='gray', linestyle=':', label=f'Óptimo={best_rp:.2f}')
ax1.set_xlabel('reg_param  (0 = QDA puro  →  1 = LDA)', fontsize=12)
ax1.set_ylabel('ROC-AUC', color='steelblue', fontsize=11)
ax2.set_ylabel('Accuracy', color='seagreen', fontsize=11)
ax1.set_title('Interpolación QDA↔LDA con reg_param — Breast Cancer',
              fontsize=13, fontweight='bold')
lines1, lab1 = ax1.get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, lab1+lab2, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# MODELO FINAL + EVALUACIÓN COMPLETA
# =============================================================================
X_tr_bc, X_te_bc, y_tr_bc, y_te_bc = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=SEED, stratify=y_bc
)
pipe_final = Pipeline([
    ('sc',  StandardScaler()),
    ('qda', QuadraticDiscriminantAnalysis(reg_param=best_rp, store_covariance=True))
])
pipe_final.fit(X_tr_bc, y_tr_bc)
y_pred_bc  = pipe_final.predict(X_te_bc)
y_proba_bc = pipe_final.predict_proba(X_te_bc)[:, 1]

print(f'QDA (reg={best_rp:.2f}) — Breast Cancer:')
print(f'  Accuracy : {accuracy_score(y_te_bc, y_pred_bc):.4f}')
print(f'  ROC-AUC  : {roc_auc_score(y_te_bc, y_proba_bc):.4f}')
print(f'  Log-loss : {log_loss(y_te_bc, y_proba_bc):.4f}')
print()
print(classification_report(y_te_bc, y_pred_bc, target_names=bc.target_names))

In [ ]:
# =============================================================================
# CURVA ROC + MATRIZ DE CONFUSIÓN
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_estimator(pipe_final, X_te_bc, y_te_bc, ax=axes[0], name='QDA')
axes[0].plot([0,1],[0,1],'k--', label='Azar')
axes[0].set_title('Curva ROC — QDA Breast Cancer', fontweight='bold')
axes[0].legend()

ConfusionMatrixDisplay(confusion_matrix(y_te_bc, y_pred_bc),
                       display_labels=bc.target_names).plot(
    ax=axes[1], colorbar=False, cmap='Purples')
axes[1].set_title('Matriz de Confusión', fontweight='bold')

plt.suptitle('QDA — Evaluación Final Breast Cancer',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# COMPARATIVA FINAL: LDA vs QDA vs LogReg vs GPC
# =============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

models_cmp = {
    'LDA':            Pipeline([('sc', StandardScaler()),
                                 ('m',  LinearDiscriminantAnalysis())]),
    'LDA (shrink)':   Pipeline([('sc', StandardScaler()),
                                 ('m',  LinearDiscriminantAnalysis(solver='lsqr',
                                                                    shrinkage='auto'))]),
    'QDA':            Pipeline([('sc', StandardScaler()),
                                 ('m',  QuadraticDiscriminantAnalysis())]),
    'QDA (reg=0.3)':  Pipeline([('sc', StandardScaler()),
                                 ('m',  QuadraticDiscriminantAnalysis(reg_param=0.3))]),
    'LogReg':         Pipeline([('sc', StandardScaler()),
                                 ('m',  LogisticRegression(C=1.0, max_iter=500))])
}

print(f'{"Modelo":<18}  {"ROC-AUC":>12}  {"Log-loss":>12}  {"Accuracy":>10}')
print('-' * 58)
for name, model in models_cmp.items():
    roc = cross_val_score(model, X_bc, y_bc, cv=skf, scoring='roc_auc')
    ll  = -cross_val_score(model, X_bc, y_bc, cv=skf, scoring='neg_log_loss')
    acc = cross_val_score(model, X_bc, y_bc, cv=skf, scoring='accuracy')
    print(f'{name:<18}  {roc.mean():.4f}±{roc.std():.3f}  '
          f'{ll.mean():.4f}±{ll.std():.3f}  {acc.mean():.4f}±{acc.std():.3f}')

---
## 10. Resumen y Checklist

| Aspecto | LDA | QDA |
|---|---|---|
| **Covarianza** | Compartida $\boldsymbol{\Sigma}$ | Por clase $\boldsymbol{\Sigma}_k$ |
| **Frontera** | Lineal | Cuadrática |
| **Parámetros** | $O(d^2 + Kd)$ | $O(Kd^2 + Kd)$ |
| **Regularización** | `shrinkage` | `reg_param` |
| **Cuándo preferir** | $n_k$ pequeño, covarianzas similares | $n_k$ grande, covarianzas distintas |

### Flujo de decisión LDA → QDA:
```
¿Supuestos gaussianos razonables?
    ├─ No → usar RF/SVM/GB
    └─ Sí → ¿Covarianzas similares por clase?
                ├─ Sí → LDA (o LDA con shrinkage si d ≈ n)
                └─ No → QDA (o QDA con reg_param si n_k es pequeño)
```

### ✅ Checklist QDA:
- [ ] ¿Tengo $n_k > d$ por cada clase? (si no, usar `reg_param > 0`)
- [ ] ¿Verifiqué que los $\det(\hat{\Sigma}_k)$ son distintos entre clases?
- [ ] ¿Busqué el `reg_param` óptimo con CV?
- [ ] ¿Inspeccioné las distancias de Mahalanobis para detectar outliers?
- [ ] ¿Comparé contra LDA con la misma CV para elegir el mejor?

---
*Próximos: Regresión Lineal, Ridge, Lasso, Elastic Net...*